# Rule Induction

**Track:** Learning | **Function:** Procedural rule discovery

Tests whether a model can discover hidden transformation rules from examples,
and whether explicit study improves performance. Single task does both
baseline and post-learning passes, logging everything to SQLite.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def extract_short_answer(response: str) -> str:
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

In [ ]:
# === Composable transformation operations ===

def op_caesar(s, k):
    return ''.join(chr((ord(c)-97+k)%26+97) if c.isalpha() else c for c in s)

def op_reverse(s, _):
    return s[::-1]

def op_swap_pairs(s, _):
    r = list(s)
    for i in range(0, len(r)-1, 2):
        r[i], r[i+1] = r[i+1], r[i]
    return ''.join(r)

def op_vowel_shift(s, _):
    m = {'a':'e','e':'i','i':'o','o':'u','u':'a'}
    return ''.join(m.get(c,c) for c in s)

def op_rotate_half(s, _):
    mid = len(s)//2
    return s[mid:] + s[:mid]

def op_consonant_shift(s, k):
    cons = 'bcdfghjklmnpqrstvwxyz'
    return ''.join(cons[(cons.index(c)+k)%len(cons)] if c in cons else c for c in s)

OPERATIONS = [
    ('caesar', op_caesar), ('reverse', op_reverse),
    ('swap_pairs', op_swap_pairs), ('vowel_shift', op_vowel_shift),
    ('rotate_half', op_rotate_half), ('consonant_shift', op_consonant_shift),
]

WORDS = [
    'planet','bridge','castle','forest','garden','hunter','jungle','knight',
    'lemon','market','orange','purple','silver','tunnel','winter','broken',
    'candle','desert','falcon','gentle','harbor','insect','jumble','kitten',
    'luster','mangle','nickel','oyster','pencil','quiver','rattle','simple',
    'temple','velvet','wander','basket','coffee','donkey',
]

def generate_rule(seed, difficulty):
    rng = random.Random(seed)
    n = {'easy':1,'medium':2,'hard':3}[difficulty]
    chosen = rng.sample(OPERATIONS, n)
    params = [rng.randint(1,5) for _ in range(n)]
    return [(name, fn, p) for (name, fn), p in zip(chosen, params)]

def apply_rule(word, rule):
    r = word.lower()
    for _, fn, p in rule:
        r = fn(r, p)
    return r

def rule_description(rule):
    return ' -> '.join(f'{name}({p})' for name, _, p in rule)

In [ ]:
# === Generate dataset: 9 items (3 seeds x 3 difficulties) ===

N_EX = {'easy': 6, 'medium': 4, 'hard': 2}
SEEDS = 3

rows = []
tid = 0

for diff in ['easy', 'medium', 'hard']:
    n_ex = N_EX[diff]
    for seed in range(SEEDS):
        rule = generate_rule(seed * 100 + hash(diff) % 1000, diff)
        rng = random.Random(seed * 7 + 13)
        words = rng.sample(WORDS, n_ex + 1)
        ex_words, test_word = words[:n_ex], words[n_ex]
        ex_text = '\n'.join(f'"{w}" -> "{apply_rule(w, rule)}"' for w in ex_words)
        expected = apply_rule(test_word, rule)
        rows.append({
            'task_id': tid, 'seed': seed, 'difficulty': diff,
            'n_examples': n_ex, 'examples_text': ex_text,
            'test_word': test_word, 'expected': expected,
            'rule_desc': rule_description(rule),
        })
        tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id', 'difficulty', 'test_word', 'expected', 'rule_desc']].to_string(index=False))

In [ ]:
# === SQLite setup (only storage during the run) ===

DB_PATH = 'rule_induction_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('''
    CREATE TABLE IF NOT EXISTS runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        model TEXT,
        task_id TEXT,
        difficulty TEXT,
        seed INTEGER,
        rule_desc TEXT,
        test_word TEXT,
        expected TEXT,
        baseline_raw TEXT,
        baseline_extracted TEXT,
        baseline_correct INTEGER,
        study_notes TEXT,
        learning_raw TEXT,
        learning_extracted TEXT,
        learning_correct INTEGER,
        baseline_time_s REAL,
        study_time_s REAL,
        learning_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready: {DB_PATH}')

In [ ]:
# === Prompt templates ===

BASELINE_P = '''I will show you examples of a secret transformation rule applied to words.
Study the examples, then apply the SAME rule to the test word.

Examples:
{examples}

Apply the same transformation to: "{test_word}"

Reply with ONLY the transformed word. No explanation, no quotes, just the word.'''

STUDY_P = '''I will show you examples of a secret transformation rule applied to words.
Analyze these examples and figure out the EXACT rule.

Examples:
{examples}

Write a detailed analysis of the transformation rule.
Describe EACH step. Test your theory against EVERY example.'''

APPLY_P = '''You previously analyzed a transformation rule and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original examples:
{examples}

Apply the rule to: "{test_word}"

Reply with ONLY the transformed word. No explanation, no quotes, just the word.'''

## Single task: baseline + learning in one pass

Does 3 LLM calls per item:
1. Baseline attempt (cold)
2. Study phase (generate notes)
3. Post-learning attempt (with notes)

Returns a dict with both scores + all raw data, logged to SQLite.

In [ ]:
@kbench.task(
    name='rule_induction',
    description='Baseline + self-learning rule induction in a single pass'
)
def rule_induction(llm, examples_text: str, test_word: str,
                   expected: str, difficulty: str, seed: int,
                   n_examples: int, rule_desc: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{difficulty}_{seed}'

    # --- Step 1: Baseline (cold attempt) ---
    t0 = time.time()
    baseline_raw = llm.prompt(BASELINE_P.format(examples=examples_text, test_word=test_word))
    baseline_time = time.time() - t0
    baseline_extracted = extract_short_answer(baseline_raw)
    baseline_correct = baseline_extracted == expected

    # --- Step 2: Study phase (generate notes) ---
    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(examples=examples_text))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    # --- Step 3: Post-learning attempt (with notes) ---
    t0 = time.time()
    learning_raw = llm.prompt(APPLY_P.format(notes=notes, examples=examples_text, test_word=test_word))
    learning_time = time.time() - t0
    learning_extracted = extract_short_answer(learning_raw)
    learning_correct = learning_extracted == expected

    # --- Log to SQLite ---
    try:
        db.execute(
            '''INSERT INTO runs (timestamp, model, task_id, difficulty, seed, rule_desc,
               test_word, expected, baseline_raw, baseline_extracted, baseline_correct,
               study_notes, learning_raw, learning_extracted, learning_correct,
               baseline_time_s, study_time_s, learning_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
            (ts, model_name, tid, difficulty, int(seed), rule_desc,
             test_word, expected, baseline_raw, baseline_extracted, int(baseline_correct),
             notes, learning_raw, learning_extracted, int(learning_correct),
             baseline_time, study_time, learning_time)
        )
        db.commit()
    except Exception as e:
        print(f'  [SQLite warn] {e}')

    # --- Print per-item detail ---
    status_b = 'Y' if baseline_correct else 'N'
    status_l = 'Y' if learning_correct else 'N'
    print(f'[{difficulty:6s}] "{test_word}" -> expected="{expected}"  '
          f'baseline="{baseline_extracted}"({status_b})  '
          f'learned="{learning_extracted}"({status_l})  '
          f'times: {baseline_time:.1f}s / {study_time:.1f}s / {learning_time:.1f}s')

    return {
        'baseline_correct': baseline_correct,
        'learning_correct': learning_correct,
        'baseline_answer': baseline_extracted,
        'learning_answer': learning_extracted,
    }


# Run evaluation
runs = rule_induction.evaluate(
    llm=[kbench.llm],
    evaluation_data=dataset.set_index('task_id'),
    n_jobs=1,
    timeout=3600,
    max_attempts=1,
)
results_df = runs.as_dataframe()
print(f'\nEvaluation complete: {len(results_df)} items')

## Results

In [ ]:
# === Read results from SQLite (canonical source) ===

results = pd.read_sql('SELECT * FROM runs ORDER BY task_id', db)
print(f'Total runs in DB: {len(results)}')
print()

# === Overall metrics ===
baseline_acc = results['baseline_correct'].mean()
learning_acc = results['learning_correct'].mean()
learning_gain = learning_acc - baseline_acc
room = 1.0 - baseline_acc
efficiency = learning_gain / room if room > 0 else 0.0

print('=' * 60)
print('RULE INDUCTION RESULTS')
print('=' * 60)
print(f'Model:                  {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Baseline accuracy:      {baseline_acc:.1%}')
print(f'Post-learning accuracy: {learning_acc:.1%}')
print(f'Learning gain:          {learning_gain:+.1%}')
print(f'Learning efficiency:    {efficiency:.1%}')
print()

# === Per-difficulty ===
print('By Difficulty:')
print('-' * 60)
for diff in ['easy', 'medium', 'hard']:
    sub = results[results['difficulty'] == diff]
    if len(sub) == 0:
        continue
    b = sub['baseline_correct'].mean()
    l = sub['learning_correct'].mean()
    g = l - b
    print(f'  {diff:8s}  baseline={b:.1%}  learned={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# === Per-item detail ===
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['baseline_correct'] else 'N'
    l = 'Y' if row['learning_correct'] else 'N'
    delta = '  HELPED' if not row['baseline_correct'] and row['learning_correct'] else \
            '  HURT' if row['baseline_correct'] and not row['learning_correct'] else ''
    print(f'  [{row["difficulty"]:6s}] "{row["test_word"]}" -> "{row["expected"]}"  '
          f'baseline="{row["baseline_extracted"]}"({b})  '
          f'learned="{row["learning_extracted"]}"({l}){delta}')

# === Timing ===
print()
print('Timing (seconds per call):')
print(f'  Baseline avg: {results["baseline_time_s"].mean():.1f}s')
print(f'  Study avg:    {results["study_time_s"].mean():.1f}s')
print(f'  Learning avg: {results["learning_time_s"].mean():.1f}s')

In [ ]:
# === Inspect study notes (why did learning help or not?) ===

print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['baseline_correct'] else 'N'
    l = 'Y' if row['learning_correct'] else 'N'
    print(f'\n--- [{row["difficulty"]}] seed={row["seed"]} test_word="{row["test_word"]}" ---')
    print(f'Rule: {row["rule_desc"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Baseline answer: "{row["baseline_extracted"]}" ({b})')
    print(f'Learning answer: "{row["learning_extracted"]}" ({l})')
    print(f'\nStudy notes (first 500 chars):')
    print(row['study_notes'][:500])
    print('...' if len(row['study_notes']) > 500 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

difficulties = ['easy', 'medium', 'hard']
baseline_scores = [results[results['difficulty'] == d]['baseline_correct'].mean() for d in difficulties]
learning_scores = [results[results['difficulty'] == d]['learning_correct'].mean() for d in difficulties]

x = range(len(difficulties))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar([i - width/2 for i in x], baseline_scores, width, label='Baseline', color='#4C72B0')
ax1.bar([i + width/2 for i in x], learning_scores, width, label='Post-Learning', color='#55A868')
ax1.set_xlabel('Difficulty')
ax1.set_ylabel('Accuracy')
ax1.set_title('Baseline vs Post-Learning Accuracy')
ax1.set_xticks(list(x))
ax1.set_xticklabels(difficulties)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [l - b for b, l in zip(baseline_scores, learning_scores)]
colors = ['#55A868' if g >= 0 else '#C44E52' for g in gains]
ax2.bar(difficulties, gains, color=colors)
ax2.set_xlabel('Difficulty')
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('rule_induction_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Post-run: upload SQLite data to BigQuery + Google Sheets ===
# Safe to pip install here — benchmark is already done, SDK can break.

results = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results)} rows to upload.\n')

# --- BigQuery ---
try:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'google-cloud-bigquery', 'db-dtypes'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    from google.cloud import bigquery
    bq = bigquery.Client()
    BQ_DATASET = 'benchmark_runs'
    BQ_TABLE = 'rule_induction'
    dataset_ref = bigquery.Dataset(f'{bq.project}.{BQ_DATASET}')
    dataset_ref.location = 'US'
    bq.create_dataset(dataset_ref, exists_ok=True)
    table_ref = f'{bq.project}.{BQ_DATASET}.{BQ_TABLE}'
    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        write_disposition='WRITE_APPEND',
    )
    job = bq.load_table_from_dataframe(results, table_ref, job_config=job_config)
    job.result()
    print(f'BigQuery: uploaded {len(results)} rows to {table_ref}')
except Exception as e:
    print(f'BigQuery upload failed (non-fatal): {e}')

# --- Google Sheets ---
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gspread'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import gspread
    from google.auth import default
    creds, _ = default(scopes=['https://www.googleapis.com/auth/spreadsheets',
                                'https://www.googleapis.com/auth/drive'])
    gc = gspread.authorize(creds)
    SHEET_NAME = 'Learning Bench — Rule Induction Runs'
    try:
        sh = gc.open(SHEET_NAME)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_NAME)
        sh.sheet1.append_row([
            'timestamp', 'model', 'task_id', 'difficulty', 'seed', 'rule_desc',
            'test_word', 'expected', 'baseline_extracted', 'baseline_correct',
            'learning_extracted', 'learning_correct',
            'baseline_time_s', 'study_time_s', 'learning_time_s',
            'study_notes_preview'
        ])
    ws = sh.sheet1
    for _, row in results.iterrows():
        ws.append_row([
            str(row['timestamp']), str(row['model']), str(row['task_id']),
            str(row['difficulty']), int(row['seed']), str(row['rule_desc']),
            str(row['test_word']), str(row['expected']),
            str(row['baseline_extracted']), int(row['baseline_correct']),
            str(row['learning_extracted']), int(row['learning_correct']),
            round(float(row['baseline_time_s']), 2),
            round(float(row['study_time_s']), 2),
            round(float(row['learning_time_s']), 2),
            str(row['study_notes'])[:200],
        ], value_input_option='RAW')
    print(f'Google Sheets: appended {len(results)} rows to "{SHEET_NAME}"')
    print(f'  URL: {sh.url}')
except Exception as e:
    print(f'Google Sheets upload failed (non-fatal): {e}')

print(f'\nDone. SQLite file: {DB_PATH}')